# Lesson 02 - Microsoft Agent Framework పరిశీలన

**Microsoft Agent Framework (MAF)** అనేది AI ఏజెంట్లను నిర్మించడానికి ఏకీకృత ఫ్రేమ్‌వర్క్. ఇది నాలుగు ప్రధాన నిర్మాణ 블ాక్‌లతో శుభ్రమైన, సంయోజనాత్మక నిర్మాణాన్ని అందిస్తుంది:

- **Client** – AI మోడల్ ఎండ్‌పాయింట్‌కి కనెక్ట్ అవుతుంది మరియు కమ్యూనికేషన్‌ను నిర్వహిస్తుంది  
- **Agent** – ఒక క్లయింట్‌ను సూచనలతో మరియు సాధన నిర్వచనాలతో చుట్టి ఉంటుంది  
- **Tools** – మోడల్ పిలవగల అనుకూల ఫంక్షన్లతో ఏజెంట్ శక్తులను విస్తరించడానికి  
- **Session** – బహు-తరం పరస్పర చర్యలకు సంభాషణ చరిత్రను నిర్వహిస్తుంది  

ఈ పాఠంలో, ఈ భావనలను ఉపయోగించి గమ్యస్థానం లభ్యతను తనిఖీ చేసే **ప్రయాణ బుకింగ్ ఏజెంట్** ను నిర్మించబోతున్నాం.


## సెటప్


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ఏజెంట్ ఫ్రేమ్‌వర్క్ ఆర్కిటెక్చర్‌ను అర్థం చేసుకోవడం

Microsoft Agent Framework ఒక పొరల ఆర్కిటెక్చర్‌ను అనుసరిస్తుంది:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **క్లయింట్** – ఒక `AzureAIProjectAgentProvider` Azure OpenAI నియంత్రణతో కనెక్ట్ అవుతుంది. ఇది ప్రామాణీకరణ, అభ్యర్థన ఆకృతీకరణ మరియు ప్రతిస్పందన పార్సింగ్ నిర్వహిస్తుంది.
2. **ఏజెంట్** – క్లయింట్ నుండి `provider.create_agent()` ద్వారా సృష్టించబడుతుంది, ఏజెంట్ మోడల్ యాక్సెస్‌ను ఆదేశాలు (సిస్టమ్ ప్రాంప్ట్) మరియు టూల్స్‌తో కలిపి ఉపయోగిస్తాడు.
3. **టూల్స్** – `@tool` తో అలంకరించబడిన Python ఫంక్షన్లు, ఏజెంట్ వాటిని చర్యలు చేపట్టడానికి లేదా డేటాను పొందడానికి పిలవగలదు.
4. **సెషన్** – ఒక `AgentSession` ఆబ్జెక్ట్ (agent.create_session() ద్వారా సృష్టించబడినది) సంభాషణ చరిత్ర నిల్వ చేస్తుంది, ఏజెంట్ మునుపటి సందర్భాన్ని గుర్తుంచుకుంటూ బహుళ-తిరుగు సంభాషణను సం�చలిస్తుంది.

ప్రతి పొరను దశలవారీగా నిర్మిద్దాం.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool డెకొరేటర్‌తో టూల్స్ జోడించడం

టూల్స్ ఏజెంట్స్‌ను టెక్స్ట్ ఉత్పత్తి దాటి చర్యలు తీసుకోవడానికి అనుమతిస్తాయి. `@tool` డెకొరేటర్ సాధారణ Python ఫంక్షన్‌ను ఏజెంట్ కాల్ చేయగలిగే ఏదోగా మార్చుతుంది.

ప్రధాన అంశాలు:
- ప్రతి పారామీటర్‌ను మోడల్ అర్థం చేసుకునేలా `Annotated[type, "description"]` ఉపయోగించండి.
- డోక్స్ట్రింగ్ మోడల్ చూసే టూల్ వివరణగా మారుతుంది.
- `approval_mode="never_require"` అంటే టూల్ యూజర్ ఆమోదం లేకుండా స్వయంచాలకంగా నడుస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## టూల్స్ తో ఏజెంట్ సృష్టించడం

ఇప్పుడు మేము క్లయింట్, సూచనలు, మరియు టూల్స్ ని ఒక ఏజెంట్ గా కలిపします. `instructions` సిస్టమ్ ప్రాంప్ట్ గా పనిచేస్తాయి — అవి ఏజెంట్ యొక్క వ్యక్తిత్వాన్ని మరియు ప్రవర్తనను నిర్వచిస్తాయి.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## సెషన్లతో మల్టీ-టర్న్ సంభాషణలు

ఒక `AgentSession` ( `agent.create_session()` ద్వారా సృష్టించబడింది) సంభాషణలో ఉన్న అన్ని సందేశాలను గమనిస్తుంది. ప్రతి `agent.run()` కాల్ కు అదే సెషనును పంపించడం ద్వారా, ఏజెంట్ కు పూర్తి సంభాషణ చరిత్రకు యాక్సెస్ కలిగి ఉంటుంది మరియు ముందు సందేశాలకు తిరిగి చూడొచ్చు.

మేము `tools=[check_destination_availability]` ను పంపుతాము కాబట్టి ఏజెంట్ ప్రతి టర్న్ లో మా అందుబాటు చెకర్ ను పిలవగలదు.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework యొక్క నాలుగు పిలర్లను పరిశీలించారు:

| కాన్సెప్ట్ | మీరు నేర్చుకున్నది |
|---------|------------------|
| **క్లయింట్** | `AzureAIProjectAgentProvider` క్రెడెన్షియల్ ఆధారిత ఆత్మీయతతో Azure OpenAI కు కనెక్ట్ అవుతుంది |
| **ఏజెంట్** | `provider.create_agent()` ఒక మోడల్ కనెక్షన్‌ను సూచనలతో మరియు ఒక పేరుతో బండిల్ చేస్తుంది |
| **సాధనాలు** | `@tool` డెకొరేటర్ ఏజెంట్ పిలవడానికి పైథాన్ ఫంక్షన్లను ప్రాప్తి తీసివేయుతుంది |
| **సెషన్** | `agent.create_session()` అనేక తిప్పుల క్రాస్ సంభాషణ చరిత్రను నిర్వహిస్తుంది |

ఈ నిర్మాణ బ్లాక్స్ కలిసి సహజ సంభాషణలు నిర్వహించే ఖాతాదారులను తయారు చేయడానికి, బయటి ఫంక్షన్లను పిలవడానికి మరియు సాందర్భాన్ని నిర్వహించడానికి ఉపయోగపడతాయి — తదుపరి పాఠాలలో మరిన్ని అభివృద్ధి చెందిన ఏజెంటిక్ నమూనాలకు పునాది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత**:  
ఈ పత్రాన్ని AI అనువాద సేవ అయిన [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించారు. మేము ఖచ్చితత్వాన్ని పాటిస్తున్నప్పటికీ, స్వయంచాలక అనువాదాల్లో తప్పులు లేదా లోపాలు ఉండవచ్చు అని దయచేసి గమనించండి. ఒరిజనల్ పత్రం native భాషలోనే అధికారిక సోర్స్‌గా తీసుకోవాలి. ముఖ్యమైన సమాచారానికి, నిపుణుల చేతి అనువాదం చేసుకోవటం సూచించబడుతుంది. ఈ అనువాదం వలన తలెత్తే ఏమైనా దుస్సమజ్ఞతలు లేదా తప్పుగా అర్థం చేసుకోవడాలపై మేము బాధ్యులు కాదు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
